# Starsim-inspired Simple SIR Disease Model

A complete, runnable Python notebook implementing a compact, agent-based style SIR-like disease simulation inspired by the Starsim framework. This notebook provides a minimal, self-contained model suitable for demonstration and experimentation in Google Colab or standard Jupyter environments.

## Disease model explanation (concise)

- We'll simulate a closed population of agents with three health states: Susceptible (S), Infected (I), and Recovered (R).
- Time advances in discrete steps (dt in years). The total simulated period is t_end years.
- Transmission: a susceptible agent becomes infected with probability depending on the fraction of currently infected agents and a transmission rate parameter beta. This is a common, simplified mass-action approach to illustrate infection spread.
- Recovery: infected agents recover with probability gamma per year.
- Assumptions and limitations (clear labeling, no clinical facts invented):
  - Closed population, no births or deaths (except optional extension can be added later).
  - Homogeneous mixing: every agent has an equal chance to contact any other agent (no explicit network structure).
  - No births, age structure, vaccination, or co-infections are included in this minimal model.
  - Stochasticity is implemented via random draws per time step per population, not per-agent pair interactions (computationally efficient approximation).
  - Initial conditions and parameters are provided in the code cells; adjust as needed. Missing values from the documentation are not required for this minimal runnable example.

## Research basis, assumptions, parameters, initial conditions, equations, and limitations (summary)

- Research basis: The Starsim notes describe an agent-based modeling framework capable of handling disease states, networks, and interventions. This notebook implements a compact, standalone, stochastic, SIR-like toy model inspired by those concepts to demonstrate a runnable simulation in Colab/Jupyter without external dependencies beyond NumPy and Matplotlib.
- Assumptions (explicit):
  - Population size N fixed; no births/deaths.
  - Homogeneous mixing; infection probability is a function of fraction infected in the population.
  - Time unit is years; dt is a fractional time step in years.
  - No co-infections, no immunity waning beyond recovered state, no vaccinations.
- Parameters (provided in code):
  - N: number of agents
  - I0: initial infected fraction or count
  - beta: transmission rate per year per infected fraction (mass-action)
  - gamma: recovery rate per year
  - t_end: total simulation time in years
  - dt: time step in years
- Initial conditions: Start with a small number of infected agents and the rest susceptible; recovered start at zero.
- Equations (discrete-time, stochastic flavor):
  - S(t+dt) = S(t) - new_infections
  - I(t+dt) = I(t) + new_infections - new_recoveries
  - R(t+dt) = R(t) + new_recoveries
  - new_infections ~ Hypergeometric-like Bernoulli draws approximated via p_infect = 1 - exp(-beta * I(t)/N * dt) for S agents
  - new_recoveries ~ Binomial(I(t), gamma * dt)
- Limitations: This is a simplified, educational example. Real Starsim is a much richer ABM with networks, multiple pathogens, and interventions.

## Setup and execution guidance (Colab / Jupyter)

1) In Google Colab:
- Ensure runtime is Python 3.x.
- Install any dependencies if needed (NumPy and Matplotlib are standard):
  - import sys; print(sys.version)
2) In standard Jupyter:
- Ensure a Python 3 kernel is active and run cells in order.

This notebook uses NumPy for numerical arrays and Matplotlib for plotting. It is self-contained and does not require additional data files.

In [ ]:
# Executable model implementation: simple SIR with homogeneous mixing
# Import necessary libraries
import numpy as np
import matplotlib.pyplot as plt

# User-definable parameters
N = 10000                 # total population
I0 = 10                   # initial infected individuals
R0 = 0                    # initial recovered individuals
beta = 0.6                # transmission rate (per year) - adjusted for dt
gamma = 0.2               # recovery rate per year
t_end = 50.0              # total simulation time in years
dt = 0.1                  # time step in years

# Derived quantities
nt = int(np.ceil(t_end / dt))
time = np.linspace(0, t_end, nt+1)

# Initial state vectors
S = np.zeros(nt+1, dtype=np.float64)
I = np.zeros(nt+1, dtype=np.float64)
R = np.zeros(nt+1, dtype=np.float64)

S[0] = max(N - I0 - R0, 0)
I[0] = I0
R[0] = R0

# Random number generator for reproducibility
rng = np.random.default_rng(seed=42)

# Time evolution (discrete step with stochastic infection probability)
for t in range(nt):
    S_t = S[t]
    I_t = I[t]
    R_t = R[t]

    if N <= 0 or S_t <= 0 and I_t <= 0:
        # no population or no infected agents left
        S[t+1] = S_t
        I[t+1] = I_t
        R[t+1] = R_t
        continue

    # Proportion infected in current step
    if N > 0:
        p_infect = 1.0 - np.exp(-beta * (I_t / N) * dt)
        p_infect = min(max(p_infect, 0.0), 1.0)
    else:
        p_infect = 0.0

    # Number of new infections drawn from a Binomial(S, p_infect) for efficiency
    new_infections = rng.binomial(int(round(S_t)), p_infect) if S_t > 0 else 0

    # Recoveries from currently infected
    new_recoveries = rng.binomial(int(round(I_t)), max(0.0, min(1.0, gamma * dt))) if I_t > 0 else 0

    # Update counts; ensure non-negative and consistent totals
    S_next = max(S_t - new_infections, 0)
    I_next = max(I_t + new_infections - new_recoveries, 0)
    R_next = max(R_t + new_recoveries, 0)
    # Keep totals aligned to N (within numerical tolerance)
    total = S_next + I_next + R_next
    if total != N:
        # adjust minor drift to keep population constant
        diff = N - total
        S_next = max(S_next + diff, 0)

    S[t+1] = S_next
    I[t+1] = I_next
    R[t+1] = R_next

# Basic visuals: S, I, R over time
plt.figure(figsize=(10,6))
plt.plot(time, S, label='Susceptible')
plt.plot(time, I, label='Infected')
plt.plot(time, R, label='Recovered')
plt.xlabel('Time (years)')
plt.ylabel('Number of individuals')
plt.title('SIR-like epidemic with homogeneous mixing (toy model)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Simple metrics
peak_infected = int(np.max(I))
peak_time = float(time[np.argmax(I)])
final_infected = int(I[-1])

print("Peak infected:", peak_infected, "at year", round(peak_time,2))
print("Final infected (end):", final_infected)
print("Final S, I, R:", int(S[-1]), int(I[-1]), int(R[-1]))


## How to interpret the outputs

- The plot shows the simulated counts of Susceptible, Infected, and Recovered individuals over time.
- The peak of the Infected curve indicates the epidemic's maximum burden under the chosen parameters.
- The final values show how much of the population remains susceptible or has progressed to recovery (and potential immunity) at the end of the simulation.
- Sensitivity: adjusting beta (transmission) or gamma (recovery) changes epidemic dynamics; this toy model is useful for quick experimentation and visualization rather than precise epidemiological forecasting.